# Channel Prediction Model Training

## Setup Instructions (First Time Only)

1. **Add GitHub Token to Colab Secrets:**
   - Click the **key icon** in the left sidebar
   - Add new secret: Name = `GITHUB_TOKEN`, Value = your token
   - Toggle the switch to enable notebook access

2. **Upload data folder to Google Drive:**
   - Upload your `data` folder (with `feature_cache` inside) to Google Drive
   - Default expected path: `MyDrive/autotrade2_data/`

3. **Select GPU Runtime:**
   - Runtime → Change runtime type → GPU (T4 is fine)

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2: Clone private repo using secret token
from google.colab import userdata
import os

token = userdata.get('GITHUB_TOKEN')
repo_url = f"https://{token}@github.com/frankywashere/autotrade2.git"

# Clone if not already cloned
if not os.path.exists('/content/autotrade2'):
    !git clone {repo_url} /content/autotrade2
else:
    print("Repo already exists, pulling latest...")
    !cd /content/autotrade2 && git pull

%cd /content/autotrade2
!git checkout x7
!git pull

In [ ]:
# Step 3: Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# Step 4: Link data folder from Google Drive
# Adjust the path if your data folder is in a different location on Drive

import os

DRIVE_DATA_PATH = "/content/drive/MyDrive/autotrade2_data"  # <-- EDIT THIS if different

# Remove existing data folder/link if exists
!rm -rf /content/autotrade2/data

# Create symlink to Drive data
!ln -s {DRIVE_DATA_PATH} /content/autotrade2/data

# Verify cache exists
cache_path = "/content/autotrade2/data/feature_cache/channel_samples.pkl"
if os.path.exists(cache_path):
    size_mb = os.path.getsize(cache_path) / (1024*1024)
    print(f"Cache found: {size_mb:.1f} MB")
else:
    print("WARNING: Cache not found! Check your Drive path.")

In [ ]:
# Step 5: Verify GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 6: Run Training!
# Edit the command below as needed

!python train.py --no-interactive \
  --mode walk-forward \
  --wf-windows 3 \
  --wf-val-months 5 \
  --wf-type expanding \
  --train-end 2022-07-08 \
  --val-end 2024-02-16 \
  --attention-heads 16 \
  --hidden-dim 128 \
  --cfc-units 256 \
  --dropout 0.3 \
  --no-shared-heads \
  --se-blocks \
  --epochs 50 \
  --batch-size 64 \
  --lr 0.001 \
  --optimizer adamw \
  --scheduler cosine_restarts \
  --weight-mode fixed_duration_focus \
  --calibration-mode brier_per_tf \
  --no-use-amp \
  --early-stopping 0 \
  --early-stopping-metric duration \
  --weight-decay 0.1 \
  --gradient-clip 1.0 \
  --uncertainty-penalty 0.1 \
  --min-duration-precision 0.25 \
  --window-strategy bounce_first \
  --device cuda

In [ ]:
# Optional: Copy results back to Drive
# Uncomment and run after training completes

# !cp -r /content/autotrade2/runs/* /content/drive/MyDrive/autotrade2_runs/